In [2]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus

In [ ]:
# parameter settings: -- parameter loading from task1

products = ['A1', 'A2', 'A3', 'B1', 'B2']

# weekly demand 
demand_data = {
    'Weekly_Mean': [
        962,  
        1923, 
        2500, 
        1154, 
        1538  
    ],
    'Weekly_CV': [0.15, 0.2, 0.2, 0.12, 0.18] 
}

demand_df = pd.DataFrame(demand_data, index=products) 

# the BOM of requirements of each Products (A, B) from each Parts (P1~P20)
bom_df = pd.read_csv('../Data/BOM.csv', index_col=0).T # for the later matrix multiplication dim align

# calculate the target capacity for products
Z_score = norm.ppf(0.995)

demand_df['Weekly_std'] = demand_df['Weekly_Mean'] * demand_df['Weekly_CV']
# round up to maintain integer
demand_df['Product_Capacity'] = np.ceil(demand_df['Weekly_Mean'] + demand_df['Weekly_std']*Z_score)


# Aggregate to parts demand
part_demand = pd.DataFrame()
part_demand['Part_Weekly_Capacity'] = bom_df.dot(demand_df['Product_Capacity'])

part_demand['Part_Weekly_Capacity'] = np.ceil(part_demand['Part_Weekly_Capacity']) 

# Fix: Strip whitespace from the index
part_demand.index = part_demand.index.str.strip()

print('weekly capacity for each product:\n', demand_df['Product_Capacity'])
print('\nweekly capacity for each part:\n', part_demand)

weekly capacity for each product:
 A1    1334.0
A2    2914.0
A3    3788.0
B1    1511.0
B2    2252.0
Name: Product_Capacity, dtype: float64

weekly capacity for each part:
      Part_Weekly_Capacity
P1                30610.0
P2                15934.0
P3                 8296.0
P4                13512.0
P5                10490.0
P6                 5838.0
P7                20664.0
P8                 6044.0
P9                13404.0
P10               18289.0
P11               11676.0
P12               14344.0
P13                7172.0
P14               26758.0
P15                3022.0
P16               18066.0
P17                7172.0
P18               17258.0
P19               29276.0
P20               21725.0


In [4]:
f_demand = part_demand / 4

## Can do different formulation